# Hindi / Hinglish decoder comparison (Project 4)

Implements `compare-hindi-model.req.md`: a reproducible comparison of small Hugging Face decoders for **Hindi in Devanagari** (`hi-Deva`) and **Romanized Hinglish** (`hinglish-Latn`), before one is chosen as the decoder of Project 4's stitched encoder-decoder. This is a model-selection experiment only: no persona training, no encoder, no cross-attention.

**What the notebook does**
1. Verifies every candidate repository on the Hub (revision, licence, access, base model, adapter configuration, architecture) and pins it to that revision. A repository that can't be fetched is marked **blocked** with the reason; nothing is substituted.
2. Runs the fixed test set (`compare-hindi-model-prompts.jsonl`: 20 prompts × 2 output modes = 40 cases) through every variant with **one generation setup**: greedy, 100 new tokens, fp32, each model's own chat template, seed 42.
3. For the Qwen2.5-1.5B Hinglish adapter: **A** base only, **B** base + adapter, **C** the adapter merged into a fresh in-memory copy; B and C are compared output by output.
4. Saves the raw outputs with their configuration, **supplementary** automatic checks (script share, repetition, clean stop), a scoring sheet with empty rubric columns for human reviewers, the decoder structure for the integration question, and a draft report.

**What it does not do:** score quality. The rubric (Section 8 of the requirement) is human review; the automatic checks are supplementary and never replace it. The draft report makes no recommendation until the scores are filled in.

**Runs on:** Kaggle, GPU T4 (one GPU is used), Internet on. **Gated models** (`google/gemma-3-1b-it`, `meta-llama/Llama-3.2-1B-Instruct`) need a Hugging Face token whose account has accepted their licences: paste it into `HF_TOKEN` in Cell 0 (and clear it before saving or sharing the notebook). Without it those two are reported as blocked. About 30–45 minutes.

**Outputs** (folder `compare-hindi-model-out/`, download from the Output panel): `compare-hindi-model-results.jsonl`, `compare-hindi-model-scores.csv`, `compare-hindi-model-report.md`, `verification.json`.

In [ ]:
# Cell 0 — Hugging Face token: paste it here before running (needed for the gated Gemma 3 and Llama 3.2; leave "" to skip them)
# Clear it again before saving or sharing the notebook, so the token never ends up in a file.

HF_TOKEN = ""

In [ ]:
# Cell 1 — Setup: packages, settings, Hugging Face login

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import csv, datetime, gc, json, os, re, statistics, subprocess, time
from pathlib import Path

import peft
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download, login

MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "100"))
SEED = 42
DTYPE = torch.float32                        # every candidate in fp32: a T4 has no fast bfloat16, and Gemma 3 is unstable in fp16
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ALLOW_REMOTE_CODE = False                    # no candidate needs custom model code (checked in Cell 3); never enabled silently
OUT = Path("compare-hindi-model-out")
OUT.mkdir(exist_ok=True)
TODAY = datetime.date.today().isoformat()


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


HF_TOKEN = HF_TOKEN.strip() or None         # from Cell 0
if HF_TOKEN:
    login(HF_TOKEN)
log(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | "
    f"{torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else 'CPU'} | HF token: {'yes' if HF_TOKEN else 'no (gated models will be blocked)'}")

## Cell 2 — The fixed test set

20 prompts (10 written in English, 10 in informal Hinglish), each asked in both output modes: 40 cases. Every case carries its `case_id`, the input exactly as written, the output mode, the instruction for that mode, the expected characteristics and the script constraint. The same 40 cases go to every variant; no persona tags.

In [ ]:
# Cell 2 — Load the 40 cases (from this folder, or from the branch on Kaggle)

DATA_BRANCH = "hindi-decoder-compare"
PROMPTS = "compare-hindi-model-prompts.jsonl"
HERE = next((d for d in (Path("."), Path("project4-hindi-decoder")) if (d / PROMPTS).exists()), None)
if HERE is None:                                              # on Kaggle: clone (or refresh) this branch
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    HERE = repo / "project4-hindi-decoder"
CASES = [json.loads(line) for line in open(HERE / PROMPTS, encoding="utf-8")]
assert len(CASES) == 40 and len({c["case_id"] for c in CASES}) == 40
print(f"{len(CASES)} cases | prompts {len({c['prompt_id'] for c in CASES})} | modes {sorted({c['output_mode'] for c in CASES})}")
for c in CASES[:2]:
    print(f"  {c['case_id']}: {c['instruction']} || {c['input']}")

## Cell 3 — Candidates, verified on the Hub

Eight variants from the six candidates. Each repository is checked now and **pinned to the revision found**: licence, gated access (can its files actually be fetched with our token?), base model, parameter count, code files, and either the adapter configuration or the model configuration (layers, hidden size, heads, key/value heads, position encoding, vocabulary) for the integration question.

| Variant | Repositories | Prompt format |
|---|---|---|
| `qwen2.5-1.5b-base` (A) | `Qwen/Qwen2.5-1.5B` | The adapter's plain ChatML (a base model has no chat format of its own), so A and B differ only by the adapter |
| `qwen2.5-1.5b+hinglish-adapter` (B) | base + `sarthaksenapati/qwen1.5b-hinglish-sft-v2` (PEFT) | The adapter's documented plain ChatML, no system prompt; its own tokenizer files |
| `qwen2.5-1.5b+hinglish-merged` (C) | the same, merged into a fresh in-memory copy | as B |
| `gemma-3-1b-it` | `google/gemma-3-1b-it` (gated) | its chat template |
| `gemma-3-1b-hindi-instruct` | `pankajpandey-dev/gemma-3-1b-hindi-instruct` | its chat template |
| `llama-3.2-1b-instruct` | `meta-llama/Llama-3.2-1B-Instruct` (gated) | its chat template |
| `qwen3-0.6b` | `Qwen/Qwen3-0.6B` | its chat template with `enable_thinking=False` (documented switch for direct replies) |
| `minicpm5-1b-hindi-instruct` | `pankajpandey-dev/MiniCPM5-1B-Hindi-Instruct` | its chat template. Its card shows `trust_remote_code=True`, but the repository has no code files and a standard Llama configuration, so it is loaded without remote code |

**Documented settings not applied** (one setup for all, as the requirement asks; recorded here as deviations): the adapter card recommends `repetition_penalty=1.2`, `no_repeat_ngram_size=3` and a 4-bit base; the Gemma Hindi card recommends sampling (`temperature=0.4`, `top_p=0.9`, `repetition_penalty=1.3`); the MiniCPM5 card shows sampling in fp16. A sampling or documented-settings pass would be a separate experiment.

In [ ]:
# Cell 3 — The variants, and the verification of every repository (pinned to the revision found)

from transformers import AutoConfig

ADAPTER = "sarthaksenapati/qwen1.5b-hinglish-sft-v2"
QWEN_BASE = "Qwen/Qwen2.5-1.5B"
CHATML = ("{% for m in messages %}{{ '<|im_start|>' + m['role'] + '\n' + m['content'] + '<|im_end|>' + '\n' }}{% endfor %}"
          "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}")     # the adapter's documented template
VARIANTS = [
    {"name": "qwen2.5-1.5b-base", "model_id": QWEN_BASE, "adapter_id": None, "tokenizer_id": QWEN_BASE, "chatml": True,
     "note": "A: base only, with the adapter's plain ChatML so that A and B differ only by the adapter"},
    {"name": "qwen2.5-1.5b+hinglish-adapter", "model_id": QWEN_BASE, "adapter_id": ADAPTER, "tokenizer_id": ADAPTER, "chatml": True,
     "note": "B: base + adapter via PeftModel, as documented (in fp32 instead of the card's 4-bit base)"},
    {"name": "qwen2.5-1.5b+hinglish-merged", "model_id": QWEN_BASE, "adapter_id": ADAPTER, "tokenizer_id": ADAPTER, "chatml": True,
     "merge": True, "note": "C: the adapter merged into a fresh in-memory copy of the base; nothing is saved or overwritten"},
    {"name": "gemma-3-1b-it", "model_id": "google/gemma-3-1b-it", "note": "gated; its chat template"},
    {"name": "gemma-3-1b-hindi-instruct", "model_id": "pankajpandey-dev/gemma-3-1b-hindi-instruct", "note": "its chat template"},
    {"name": "llama-3.2-1b-instruct", "model_id": "meta-llama/Llama-3.2-1B-Instruct", "note": "gated; its chat template"},
    {"name": "qwen3-0.6b", "model_id": "Qwen/Qwen3-0.6B", "template_kwargs": {"enable_thinking": False},
     "note": "its chat template with thinking off"},
    {"name": "minicpm5-1b-hindi-instruct", "model_id": "pankajpandey-dev/MiniCPM5-1B-Hindi-Instruct",
     "note": "its chat template; loaded without remote code (no code files in the repository)"},
]
CONFIG_KEYS = ["architectures", "model_type", "num_hidden_layers", "hidden_size", "intermediate_size", "num_attention_heads",
               "num_key_value_heads", "head_dim", "hidden_act", "max_position_embeddings", "rope_theta", "rope_parameters", "rope_scaling",
               "sliding_window", "rms_norm_eps", "vocab_size", "tie_word_embeddings"]
api = HfApi()


def verify(repo):
    try:
        info = api.model_info(repo, token=HF_TOKEN)
    except Exception as e:
        return {"repo": repo, "status": "blocked", "reason": f"{type(e).__name__}: {e}"[:300]}
    files = [s.rfilename for s in info.siblings]
    card = info.card_data.to_dict() if info.card_data else {}
    rec = {"repo": repo, "status": "ok", "revision": info.sha, "card_url": f"https://huggingface.co/{repo}", "checked": TODAY,
           "license": card.get("license"), "gated": info.gated, "base_model": card.get("base_model"), "library": info.library_name,
           "params": info.safetensors.total if info.safetensors else None, "code_files": [f for f in files if f.endswith(".py")],
           "is_adapter": "adapter_config.json" in files}
    try:                                                      # can the files actually be fetched (gated repos need an approved token)?
        path = hf_hub_download(repo, "adapter_config.json" if rec["is_adapter"] else "config.json", revision=info.sha, token=HF_TOKEN)
    except Exception as e:
        rec.update(status="blocked", reason=f"{type(e).__name__}: {e}"[:300])
        return rec
    if rec["is_adapter"]:
        a = json.load(open(path))
        rec["adapter"] = {k: a.get(k) for k in ("base_model_name_or_path", "peft_type", "r", "lora_alpha", "lora_dropout",
                                                "target_modules", "task_type", "modules_to_save")}
    else:
        cfg = AutoConfig.from_pretrained(repo, revision=info.sha, token=HF_TOKEN).to_dict()
        rec["config"] = {k: cfg.get(k) for k in CONFIG_KEYS if cfg.get(k) is not None}
    return rec


VERIFY = {r: verify(r) for r in dict.fromkeys(x for v in VARIANTS for x in (v["model_id"], v.get("adapter_id")) if x)}
if VERIFY[ADAPTER]["status"] == "ok" and VERIFY[ADAPTER]["adapter"]["base_model_name_or_path"] != QWEN_BASE:
    VERIFY[ADAPTER].update(status="blocked", reason=f"adapter targets {VERIFY[ADAPTER]['adapter']['base_model_name_or_path']}, not {QWEN_BASE}")
json.dump(VERIFY, open(OUT / "verification.json", "w"), indent=1, default=str)

print(f"{'repository':46s} | {'status':7s} | {'revision':10s} | {'licence':11s} | {'gated':6s} | {'params':>6s} | type / base")
for r, x in VERIFY.items():
    kind = (f"adapter {x['adapter']['peft_type']} r{x['adapter']['r']} on {x['adapter']['base_model_name_or_path']}" if x.get("adapter")
            else f"{(x.get('config') or {}).get('architectures')} | base {x.get('base_model')}")
    params = f"{x['params'] / 1e9:.2f}B" if x.get("params") else "—"
    print(f"{r:46s} | {x['status']:7s} | {(x.get('revision') or '—')[:10]:10s} | {str(x.get('license')):11s} | {str(x.get('gated')):6s} | "
          f"{params:>6s} | {kind}" + (f" | code files {x['code_files']}" if x.get("code_files") else "")
          + (f"\n{'':46s}   reason: {x['reason']}" if x["status"] != "ok" else ""))

## Cell 4 — Prompt and generation

The user turn is the mode's instruction, a blank line, then the input exactly as written. The chat template turns it into the model's prompt; the template already holds any start token, so the prompt is tokenized without adding special tokens. Generation: greedy (`do_sample=False`, one beam, no repetition penalty), at most 100 new tokens, stopping at the model's own end tokens (for ChatML also `<|im_end|>`). Only the **new** tokens are decoded, so the prompt can't leak into the saved answer. Every output is timed with the GPU synchronized.

In [ ]:
# Cell 4 — Load a variant, build its prompt, generate one answer

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN = {"do_sample": False, "num_beams": 1, "max_new_tokens": MAX_NEW_TOKENS, "repetition_penalty": 1.0}


def rev(repo):
    return VERIFY[repo]["revision"]


def load_variant(v):
    tok = AutoTokenizer.from_pretrained(v.get("tokenizer_id", v["model_id"]), revision=rev(v.get("tokenizer_id", v["model_id"])), token=HF_TOKEN)
    if v.get("chatml"):
        tok.chat_template = CHATML
    model = AutoModelForCausalLM.from_pretrained(v["model_id"], revision=rev(v["model_id"]), dtype=DTYPE, token=HF_TOKEN,
                                                 trust_remote_code=ALLOW_REMOTE_CODE).to(DEVICE)
    if v.get("adapter_id"):
        model = PeftModel.from_pretrained(model, v["adapter_id"], revision=rev(v["adapter_id"]), token=HF_TOKEN)
        if v.get("merge"):
            model = model.merge_and_unload()                  # in memory only: no weights are written anywhere
    model.eval()
    eos = model.generation_config.eos_token_id
    eos = set(eos if isinstance(eos, list) else [eos]) | {tok.eos_token_id}
    if v.get("chatml"):
        eos |= {tok.convert_tokens_to_ids("<|im_end|>")}
    eos = sorted(e for e in eos if e is not None)
    pad = tok.pad_token_id if tok.pad_token_id is not None else eos[0]
    return tok, model, eos, pad


def prompt_text(tok, v, case):
    msgs = [{"role": "user", "content": f"{case['instruction']}\n\n{case['input']}"}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, **v.get("template_kwargs", {}))


def generate(tok, model, eos, pad, text):
    ids = tok(text, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    torch.manual_seed(SEED)
    torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**ids, **GEN, eos_token_id=eos, pad_token_id=pad, temperature=None, top_p=None, top_k=None)
    torch.cuda.synchronize(DEVICE)
    ms = (time.perf_counter() - t0) * 1000
    new = out[0, ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), {
        "prompt_tokens": int(ids["input_ids"].shape[1]), "new_tokens": int(new.numel()),
        "stopped_cleanly": bool(new.numel() and new[-1].item() in eos), "ms": round(ms)}

---
## Cell 5 — Run every variant on all 40 cases

One variant at a time on `cuda:0`, freed before the next. A blocked repository or a load failure is recorded against every case of that variant (status `blocked` or `load_failed`), and a generation error against its case: failed runs stay in the results. Each row has the fields the requirement lists, plus the full prompt and the token counts.

In [ ]:
# Cell 5 — Generate: 8 variants x 40 cases, saved to compare-hindi-model-results.jsonl as they finish

RESULTS, RUNINFO = [], {}
results_file = OUT / "compare-hindi-model-results.jsonl"
results_file.write_text("")


def record(v, case, status, output=None, error=None, text=None, stats=None, template=None):
    row = {"case_id": case["case_id"], "prompt_id": case["prompt_id"], "variant": v["name"], "model_id": v["model_id"],
           "model_revision": VERIFY[v["model_id"]].get("revision"), "adapter_id": v.get("adapter_id"),
           "adapter_revision": VERIFY[v["adapter_id"]].get("revision") if v.get("adapter_id") else None,
           "output_mode": case["output_mode"], "input": case["input"], "prompt": text, "generated_output": output,
           "generation_config": {**GEN, "dtype": str(DTYPE), "seed": SEED, "template": template,
                                 "template_kwargs": v.get("template_kwargs", {}), "merged": bool(v.get("merge"))},
           **(stats or {}), "status": status, "error": error}
    RESULTS.append(row)
    with open(results_file, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")


for v in VARIANTS:
    repos = [v["model_id"]] + ([v["adapter_id"]] if v.get("adapter_id") else [])
    blocked = [f"{r}: {VERIFY[r].get('reason')}" for r in repos if VERIFY[r]["status"] != "ok"]
    if blocked:
        log(f"{v['name']}: BLOCKED ({'; '.join(blocked)})")
        for case in CASES:
            record(v, case, "blocked", error="; ".join(blocked))
        RUNINFO[v["name"]] = {"status": "blocked"}
        continue
    try:
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats(DEVICE)
        t0 = time.perf_counter()
        tok, model, eos, pad = load_variant(v)
        load_s = time.perf_counter() - t0
    except Exception as e:
        log(f"{v['name']}: LOAD FAILED: {type(e).__name__}: {e}")
        for case in CASES:
            record(v, case, "load_failed", error=f"{type(e).__name__}: {e}"[:500])
        RUNINFO[v["name"]] = {"status": "load_failed"}
        continue
    template = "adapter ChatML" if v.get("chatml") else "model chat template"
    log(f"{v['name']}: loaded in {load_s:.0f}s | eos {eos} | {template}")
    for case in CASES:
        text = prompt_text(tok, v, case)
        try:
            output, stats = generate(tok, model, eos, pad, text)
            record(v, case, "ok", output=output, text=text, stats=stats, template=template)
        except Exception as e:
            record(v, case, "generation_failed", error=f"{type(e).__name__}: {e}"[:500], text=text, template=template)
    done = [r for r in RESULTS if r["variant"] == v["name"] and r["status"] == "ok"]
    RUNINFO[v["name"]] = {"status": "ran", "load_s": round(load_s), "ok": len(done),
                          "peak_gpu_gb": round(torch.cuda.max_memory_allocated(DEVICE) / 1e9, 2) if DEVICE.type == "cuda" else None,
                          "median_ms": round(statistics.median(r["ms"] for r in done)) if done else None,
                          "median_new_tokens": statistics.median(r["new_tokens"] for r in done) if done else None}
    log(f"{v['name']}: {RUNINFO[v['name']]}")
    del model
    gc.collect()
    torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

print(f"\n{len(RESULTS)} rows saved to {results_file} | " + ", ".join(f"{s} {sum(r['status'] == s for r in RESULTS)}"
                                                                for s in ("ok", "blocked", "load_failed", "generation_failed")))

## Cell 6 — A look at the outputs, and B against C

Two cases per variant (one per mode) to see the outputs straight away, then the adapter check of Section 9: with the same inference settings, does the merged copy (C) give the same answers as base + adapter (B)? Small differences can come from the order of floating-point operations.

In [ ]:
# Cell 6 — Sample outputs per variant; adapter (B) vs merged (C)

for v in VARIANTS:
    for cid in ("p02-hinglish-Latn", "p13-hi-Deva"):
        r = next(r for r in RESULTS if r["variant"] == v["name"] and r["case_id"] == cid)
        shown = r["generated_output"] if r["status"] == "ok" else f"[{r['status']}] {r['error']}"
        print(f"{v['name']:30s} {cid:18s} | {shown[:300]!r}")
    print()

by = {(r["variant"], r["case_id"]): r for r in RESULTS}
pairs = [(by[("qwen2.5-1.5b+hinglish-adapter", c["case_id"])], by[("qwen2.5-1.5b+hinglish-merged", c["case_id"])]) for c in CASES]
pairs = [(b, c) for b, c in pairs if b["status"] == c["status"] == "ok"]
same = sum(b["generated_output"] == c["generated_output"] for b, c in pairs)
B_VS_C = {"compared": len(pairs), "identical": same}
print(f"adapter (B) vs merged (C): {same} / {len(pairs)} outputs identical")
for b, c in [(b, c) for b, c in pairs if b["generated_output"] != c["generated_output"]][:3]:
    print(f"  {b['case_id']}:\n    B: {b['generated_output'][:160]!r}\n    C: {c['generated_output'][:160]!r}")

---
## Cell 7 — Supplementary automatic checks

**These do not score quality** and never replace the human rubric; they only flag things worth a look:
- **Devanagari share:** Devanagari letters / (Devanagari + Latin letters). `hi-Deva` should be high (≥ 0.8 here), `hinglish-Latn` should be 0 (the check allows ≤ 0.05).
- **Repetition:** the most repeated word 3-gram (> 2 suggests a loop) and the share of distinct words.
- **Clean stop:** whether generation ended with an end token instead of hitting the 100-token cap.

In [ ]:
# Cell 7 — Script share, repetition and clean stop, per variant and output mode

DEVA, LATN = re.compile(r"[ऀ-ॿ]"), re.compile(r"[A-Za-z]")


def auto_checks(row):
    t = row["generated_output"] or ""
    d, l = len(DEVA.findall(t)), len(LATN.findall(t))
    share = d / (d + l) if d + l else None
    words = t.split()
    grams = [" ".join(words[i:i + 3]) for i in range(len(words) - 2)]
    top = max((grams.count(g) for g in set(grams)), default=0)
    ok_script = share is not None and (share >= 0.8 if row["output_mode"] == "hi-Deva" else share <= 0.05)
    return {"devanagari_share": None if share is None else round(share, 3), "script_ok_auto": ok_script, "top_trigram_repeats": top,
            "distinct_word_share": round(len(set(words)) / len(words), 3) if words else None, "words": len(words)}


for r in RESULTS:
    r["auto"] = auto_checks(r) if r["status"] == "ok" else None

print("SUPPLEMENTARY ONLY: not a quality score")
print(f"{'variant':30s} | {'mode':13s} | {'ok':>5} | {'script ok':>9} | {'Deva share':>10} | {'loops':>5} | {'clean stop':>10} | {'words':>5} | {'p50 ms':>6}")
AUTO = {}
for v in VARIANTS:
    for mode in ("hi-Deva", "hinglish-Latn"):
        rs = [r for r in RESULTS if r["variant"] == v["name"] and r["output_mode"] == mode]
        ok = [r for r in rs if r["status"] == "ok"]
        if not ok:
            AUTO[(v["name"], mode)] = None
            print(f"{v['name']:30s} | {mode:13s} | {0:>2} / {len(rs):<2} | {'—':>9} | {'—':>10} | {'—':>5} | {'—':>10} | {'—':>5} | {'—':>6}")
            continue
        shares = [r["auto"]["devanagari_share"] for r in ok if r["auto"]["devanagari_share"] is not None]
        a = {"ok": len(ok), "n": len(rs), "script_ok": sum(r["auto"]["script_ok_auto"] for r in ok),
             "deva_share": round(statistics.mean(shares), 2) if shares else None,
             "loops": sum(r["auto"]["top_trigram_repeats"] > 2 for r in ok), "clean_stop": sum(r["stopped_cleanly"] for r in ok),
             "words": round(statistics.mean(r["auto"]["words"] for r in ok)), "p50_ms": round(statistics.median(r["ms"] for r in ok))}
        AUTO[(v["name"], mode)] = a
        print(f"{v['name']:30s} | {mode:13s} | {a['ok']:>2} / {a['n']:<2} | {a['script_ok']:>4} / {a['ok']:<3} | {str(a['deva_share']):>10} | "
              f"{a['loops']:>5} | {a['clean_stop']:>4} / {a['ok']:<4} | {a['words']:>5} | {a['p50_ms']:>6}")

## Cell 8 — Scoring sheet for the human review

`compare-hindi-model-scores.csv`: one row per variant and case, with the output, the expected characteristics and the automatic checks, and **empty columns for the seven rubric dimensions** (1–5), a short rationale and the reviewer's name. Code-switching applies to `hinglish-Latn` only (write `n/a` for `hi-Deva`). Failed runs are listed too, so they stay visible. For two reviewers, copy the file and add a reviewer name to each copy.

In [ ]:
# Cell 8 — Write the scoring sheet with empty rubric columns

RUBRIC = ["naturalness", "semantic_correctness", "script_compliance", "code_switching", "grammar_readability",
          "repetition_degeneration", "instruction_following"]
case_by_id = {c["case_id"]: c for c in CASES}
with open(OUT / "compare-hindi-model-scores.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["case_id", "variant", "output_mode", "status", "input", "expected", "generated_output", "devanagari_share",
                "script_ok_auto", "top_trigram_repeats", "stopped_cleanly"] + RUBRIC + ["rationale", "reviewer"])
    for r in RESULTS:
        a = r.get("auto") or {}
        w.writerow([r["case_id"], r["variant"], r["output_mode"], r["status"], r["input"], case_by_id[r["case_id"]]["expected"],
                    r["generated_output"] if r["status"] == "ok" else f"[{r['status']}] {r['error']}",
                    a.get("devanagari_share"), a.get("script_ok_auto"), a.get("top_trigram_repeats"), r.get("stopped_cleanly")]
                   + [("n/a" if (k == "code_switching" and r["output_mode"] == "hi-Deva") else "") for k in RUBRIC] + ["", ""])
print(f"scoring sheet: {OUT / 'compare-hindi-model-scores.csv'} ({len(RESULTS)} rows, rubric columns empty for the reviewers)")

## Cell 9 — Decoder structure, for the stitched architecture

From each model's configuration (Cell 3) and this run: layers, hidden size, attention heads, key/value heads (grouped-query attention when fewer than the heads), feed-forward size, position encoding (RoPE base), vocabulary, tied embeddings, and the measured fp32 peak GPU memory and median time per answer on the T4. All candidates are standard Hugging Face causal decoders (RMSNorm, pre-norm residual blocks, rotary positions, `generate()` with a KV cache), so cross-attention would be added as new modules inside or after each decoder layer. Any custom code would show up in Cell 3 as code files. The hidden size is what the projection bridge must map the encoder into.

In [ ]:
# Cell 9 — Integration facts per model, with measured memory and latency

print(f"{'variant':30s} | {'layers':>6} | {'hidden':>6} | {'heads':>5} | {'kv':>3} | {'ffn':>6} | {'rope base':>10} | {'vocab':>7} | {'tied':>5} | "
      f"{'peak GB':>7} | {'p50 ms':>6}")
INTEGRATION = {}
for v in VARIANTS:
    c = VERIFY[v["model_id"]].get("config") or {}
    rope = c.get("rope_theta") or c.get("rope_parameters") or c.get("rope_scaling")      # transformers 5 keeps RoPE settings in rope_parameters
    info = RUNINFO.get(v["name"], {})
    INTEGRATION[v["name"]] = {**{k: c.get(k) for k in ("num_hidden_layers", "hidden_size", "num_attention_heads", "num_key_value_heads",
                                                       "intermediate_size", "vocab_size", "tie_word_embeddings", "model_type")},
                              "rope": rope, "peak_gpu_gb": info.get("peak_gpu_gb"), "median_ms": info.get("median_ms")}
    i = INTEGRATION[v["name"]]
    print(f"{v['name']:30s} | {str(i['num_hidden_layers']):>6} | {str(i['hidden_size']):>6} | {str(i['num_attention_heads']):>5} | "
          f"{str(i['num_key_value_heads']):>3} | {str(i['intermediate_size']):>6} | {str(rope)[:10]:>10} | {str(i['vocab_size']):>7} | "
          f"{str(i['tie_word_embeddings']):>5} | {str(i['peak_gpu_gb']):>7} | {str(i['median_ms']):>6}")

## Cell 10 — Draft report

Writes `compare-hindi-model-report.md` from this run: verification (documented claims), run status, the supplementary checks per mode, the B / C comparison, the integration table and two raw outputs per variant. It states plainly that the rubric scores are pending, and it makes **no recommendation**: that comes after the human review, using the decision rule of the requirement.

In [ ]:
# Cell 10 — Draft report (documented claims vs measured results; recommendation pending the human scores)

L = ["# Hindi / Hinglish decoder comparison: draft report", "",
     f"Generated by `compare-hindi-model.ipynb` on {TODAY}. **Status: outputs generated; the rubric scores (`compare-hindi-model-scores.csv`) "
     f"are pending human review, so no recommendation is made yet.** Automatic checks below are supplementary only.", "",
     f"Setup: greedy decoding, {MAX_NEW_TOKENS} new tokens, {DTYPE}, seed {SEED}, one T4; transformers {transformers.__version__}, "
     f"peft {peft.__version__}. 20 prompts x 2 modes = 40 cases per variant.", "",
     "## 1. Candidate verification (documented, from the Hub)", "",
     "| Repository | Status | Revision | Licence | Gated | Parameters | Type / base | Notes |", "|---|---|---|---|---|---|---|---|"]
for r, x in VERIFY.items():
    kind = (f"LoRA r{x['adapter']['r']} α{x['adapter']['lora_alpha']} on `{x['adapter']['base_model_name_or_path']}`" if x.get("adapter")
            else f"{(x.get('config') or {}).get('model_type')}; base {x.get('base_model')}")
    L.append(f"| `{r}` | {x['status']} | {(x.get('revision') or '—')[:12]} | {x.get('license')} | {x.get('gated')} | "
             f"{(str(round(x['params'] / 1e9, 2)) + 'B') if x.get('params') else '—'} | {kind} | {x.get('reason', '') or ''} |")
L += ["", "## 2. Runs (measured)", "", "| Variant | Status | OK cases | Load (s) | Peak GPU (GB) | Median ms per answer | Note |", "|---|---|---|---|---|---|---|"]
for v in VARIANTS:
    i = RUNINFO.get(v["name"], {})
    L.append(f"| {v['name']} | {i.get('status')} | {i.get('ok', 0)} / {len(CASES)} | {i.get('load_s', '—')} | {i.get('peak_gpu_gb', '—')} | "
             f"{i.get('median_ms', '—')} | {v['note']} |")
L += ["", f"Adapter check (Section 9): base + adapter (B) and merged (C) gave identical outputs on {B_VS_C['identical']} / {B_VS_C['compared']} cases.", "",
      "## 3. Supplementary automatic checks (not quality scores)", "",
      "| Variant | Mode | OK | Script OK (auto) | Devanagari share | Possible loops | Clean stop | Mean words |", "|---|---|---|---|---|---|---|---|"]
for (name, mode), a in AUTO.items():
    L.append(f"| {name} | {mode} | " + ("— | — | — | — | — | — |" if a is None else
             f"{a['ok']} / {a['n']} | {a['script_ok']} / {a['ok']} | {a['deva_share']} | {a['loops']} | {a['clean_stop']} / {a['ok']} | {a['words']} |"))
L += ["", "## 4. Decoder structure (from configurations; memory and latency measured)", "",
      "| Variant | Layers | Hidden | Heads | KV heads | FFN | RoPE settings | Vocabulary | Tied embeddings |", "|---|---|---|---|---|---|---|---|---|"]
for name, i in INTEGRATION.items():
    L.append(f"| {name} | {i['num_hidden_layers']} | {i['hidden_size']} | {i['num_attention_heads']} | {i['num_key_value_heads']} | "
             f"{i['intermediate_size']} | {str(i['rope'])[:80]} | {i['vocab_size']} | {i['tie_word_embeddings']} |")
L += ["", "## 5. Raw outputs (two per variant, unedited)", ""]
for v in VARIANTS:
    for cid in ("p02-hinglish-Latn", "p13-hi-Deva"):
        r = by[(v["name"], cid)]
        L.append(f"- **{v['name']}**, `{cid}`: " + (f"“{r['generated_output']}”" if r["status"] == "ok" else f"[{r['status']}]"))
L += ["", "## 6. Pending", "",
      "- Fill in the rubric columns of `compare-hindi-model-scores.csv` (two reviewers on a subset), then report per-dimension averages per variant and mode, per-prompt results, and strong / typical / weak examples.",
      "- Then apply the decision rule (Section 13 of the requirement), weighing language quality together with the decoder structure above, licence and hardware.",
      "- Documented settings not applied in this pass (adapter: repetition penalty and n-gram blocking; Gemma Hindi and MiniCPM5: sampling) belong to a separate run."]
(OUT / "compare-hindi-model-report.md").write_text("\n".join(L), encoding="utf-8")
print(f"draft report: {OUT / 'compare-hindi-model-report.md'} ({len(L)} lines)")
print("files:", sorted(p.name for p in OUT.iterdir()))

---
## Next

1. Download `compare-hindi-model-out/` from the Output panel.
2. Score the outputs in `compare-hindi-model-scores.csv` with the rubric (1–5, rationale), with two reviewers on a subset.
3. Report per-dimension averages per variant and per mode, and choose the decoder with the decision rule of the requirement: language quality in the target mode, instruction following, script, transparency, fit with the stitched architecture, licence and hardware.